# Review of the available datasets

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import io
import base64
import json
import os
import jq

import kagglehub
from datasets import load_dataset
from datasets import load_dataset_builder


import zenodo_get

from urllib.parse import urlparse, parse_qs
import urllib3

from pyzotero import zotero


import requests
import time

## Connect to Zotero

In [ ]:
# 1. Connection settings
# Replace with your Zotero User ID and API key.
# For local access, you don't need a public API key, but a private one is recommended for write permissions.
# The `local=True` parameter is crucial for connecting to your local Zotero instance.



with open("secrets.json", 'r') as f:
    secrets = json.load(f)

zs = secrets["zotero"]

USER_ID = zs["USER_ID"]
API_KEY = zs["API_KEY"]

zot = zotero.Zotero(USER_ID, 'user', API_KEY, local=True)
zot

In [ ]:
# 2. Function to get items by query
def get_zotero_items(query_string, limit=10):
    """Retrieves Zotero items matching a quick search query."""
    print(f"Searching for items with query: '{query_string}'...")
    # The `q` parameter performs a quick search
    # and `qmode` specifies what to search against.
    try:
        items = zot.items(q=query_string, qmode='everything', limit=limit)
        return items
    except Exception as e:
        print(f"An error occurred while searching: {e}")
        return []

# 3. Function to process an item's fields
def process_item_fields(item):
    """Processes an item's metadata and returns a string."""
    title = item['data'].get('title', 'No Title')
    creators = item['data'].get('creators', [])
    author_list = [f"{c['firstName']} {c['lastName']}" for c in creators if c['creatorType'] == 'author']
    authors = ", ".join(author_list) if author_list else "Unknown Author"
    
    # A simple processing example: formatting a summary string
    processed_text = (
        f"Zotero Item Summary\n"
        f"-------------------\n"
        f"Title: {title}\n"
        f"Authors: {authors}\n"
        f"Processed on: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}\n"
    )
    return processed_text.encode('utf-8')

# 4. Function to add processed data as an attachment
def add_attachment_to_item(item_key, processed_data):
    """Adds a byte stream as a new attachment file to a Zotero item."""
    print(f"Adding attachment to item with key: {item_key}...")
    try:
        # Use an in-memory buffer to simulate a file
        buf = io.BytesIO(processed_data)
        file_name = f"summary_{item_key}.txt"

        # `zot.attachment_both` uploads the file and creates a new item with the file as an attachment
        # The `parentItem` parameter links it to the original item
        result = zot.attachment_both(
            parentItem=item_key,
            item_data=zot.item_template('attachment'),
            file=buf,
            filename=file_name
        )
        print("Attachment added successfully.")
        return result
    except Exception as e:
        print(f"An error occurred while adding the attachment: {e}")
        return None

In [ ]:
# if __name__ == "__main__":
#     # Define a user query
#     user_query = "python"
    
#     # Step 1 & 2: Get items
#     my_items = get_zotero_items(user_query)

#     if not my_items:
#         print("No items found. Exiting.")
#     else:
#         print(my_items)
        # for item in my_items:
        #     item_key = item['data']['key']
        #     print(f"\nProcessing item with key: {item_key}")
            
        #     # Step 3: Process fields
        #     processed_content = process_item_fields(item)
            
        #     # Step 4 & 5: Add attachment
        #     add_attachment_to_item(item_key, processed_content)

In [ ]:
zot.collections()

In [ ]:
# Get a list of all your top-level collections
collections = zot.collections()

# Find the key for a specific collection by name
collection_key = None
for col in collections:
    if col['data']['name'] == "Datasets review":
        collection_key = col['data']['key']
        break

if collection_key:
    print(f"Found key for 'Datasets review': {collection_key}")
else:
    print("Collection not found. Please check the name.")

In [ ]:
collection_key

In [ ]:
def jload(fn = "secrets.json"):
    with open(fn, 'r') as f:
        return json.load(f)

def jsave(tree, fn):
    with open(fn, 'w+') as f:
        return json.dump(tree, f, indent=2)

In [ ]:
def md_save(df, fn):
    mcw = [10] * 5 + [None] + [40,40]
    with open(fn,'w+') as f:
        df.to_markdown(f, index = False, tablefmt="pipe", maxcolwidths=mcw)

In [ ]:
#q="",  tag="", limit=10
articles = zot.items(qmode="everything", itemType="dataset", 
                     collection="T8YELS8J", fields="citationKey")
print(len(articles))

In [ ]:
articles = zot.collection_items("T8YELS8J", qmode="everything", itemType="dataset", 
                      fields="citationKey")

print(len(articles))

In [ ]:
jsave(articles, "articles.json")

In [ ]:
jq_query = "map({key: .data.key, title: .data.title, \
                 extra: .data.extra, itemType: .data.itemType, \
                 url: .data.url, repository: .data.repository })"
result = jq.compile(jq_query).input(articles)
result = result.all()[0]
result

### Get Articles

In [ ]:
dss = pd.DataFrame(result)
dss["citekey"] = dss.extra.apply(lambda r: r.split("Citation Key: ")[1])
dss.drop("extra",axis=1, inplace=True)
dss.sort_values("repository")

## Read the Datasets

In [ ]:
def read_yaml(file):
    import yaml
    with open(file, 'r') as f:
        return pd.json_normalize(yaml.safe_load(f))

In [ ]:
# read_yaml(pth) runs too long

In [ ]:
import warnings

def read_any(file):
    if file.endswith(('.csv', 'tsv')) :
        df = pd.read_csv(file, sep=None, engine='python')
    elif file.endswith('.json'):
        df = pd.read_json(file)
    elif file.endswith('.jsonl'):
        df = pd.read_json(file, lines=True)
    elif file.endswith('.yaml'):
        df = read_yaml(file)
    elif file.endswith('.xml'):
        df = pd.read_xml(file)
    elif file.endswith(('.xls','xlsx')):
        df = pd.read_excel(file)
    elif file.endswith('.hdf'):
        df = pd.read_hdf(file)           
    elif file.endswith('.sql'):
        df = pd.read_sql(file)
    else:        
        warnings.warn(f'Skipping Unsupported filetype: {file}')
        df = None
    return df


In [ ]:
#read_any(pth)

## Download datasets

In [ ]:

def download_webpage(url, filename ):
    if os.path.exists(filename):
        print (f"File {filename} already exists, skipping download.")
        return 'skip'
        
    time.sleep(.5)  # Sleep for 0.5 second to avoid overwhelming the server
    response = requests.get(url)

    if response.status_code == 200:  # Check if the page was successfully fetched
        with open(filename, "w") as f:
            f.write(response.text)
        print(f"Downloaded {url} to {filename}")
        return 'ok'
    
    else:        
        newurl = url.replace("_all","")
        if newurl != url:
            print(f"Failed to retrieve {url}, trying alternative")
            return download_webpage(newurl, filename)
        else:
            print(f"Failed to retrieve {url} and no alternative available.")
            return 'fail'        



In [ ]:
def EXPLOSON(df, which_col = 'labels'):
    # specific to EduRABSA_Dataset
    #df = pd.read_json(fl, lines=True)
    df = df.explode(which_col)\
        .reset_index(drop=False, names = ["review_id"])
    tp = pd.json_normalize(df[which_col])
    df = pd.merge(df, tp, left_index=True, right_index=True)\
        .drop(which_col, axis=1)
    return df

need_preprocess = {"Yhua219EduRABSA_ASQEDatasets2025": lambda d: EXPLOSON(d, 'output')}


In [ ]:

tst = read_any(fl)
tst

In [ ]:
tst = EXPLOSON(tst, 'labels')


In [ ]:



def download(item):
    # Parse the URL into its components
    parsed_url = urlparse(item.url)
    
    # Access individual components
    scheme = parsed_url.scheme
    domain = parsed_url.hostname
    netloc = parsed_url.netloc
    path = parsed_url.path
    query = parsed_url.query
    fragment = parsed_url.fragment

    
    print("downloading", item.url)


    dfs = {}
    match item.repository.lower() :
        case "kaggle":
            ds_path = path.replace("/datasets/", "")
            loc_path = kagglehub.dataset_download(ds_path)
            print("downloaded to", loc_path)

            dfs = load_dir(loc_path)
                    
        case "huggingface":
            ds_path = path.replace("/datasets/", "")            
            builder = load_dataset_builder(ds_path)
            builder.download_and_prepare()
            loc_path = builder.cache_dir
            print("downloaded to", loc_path)
            files = os.listdir(loc_path)
            print("files:", files)
            dfs = builder.as_dataset()
            dfs = {k: v.to_pandas() for k,v in dfs.items()}
            #files = list(dfs.keys())
            #dfs[0] = ds["train"]

        case "lis-lab.fr":
            # TODO: download all htmls from page
            loc_path = os.path.expanduser(f"~/.cache/{item.repository.lower()}/{item.citekey}")
            url = "https://pageperso.lis-lab.fr/ismail.badache/Reviews_ExtracTerms%20HTML/Aspect%20Terms%20Sentiment%20M1%20NLTK/Reviews_ExtracTerm_C_Sent_Assignment.html"
            os.makedirs(loc_path, exist_ok=True)
            filename = "Reviews_ExtracTerm_C_Sent_Assignment.html"
            dest = os.path.join(loc_path , filename)
            status = download_webpage(url, dest)
            if status != 'fail':
                dfs[filename] = pd.read_html(dest)[0]

        case "zenodo":
            ds_path = path.replace("/records/", "")            
            loc_path = os.path.expanduser(f"~/.cache/{item.repository.lower()}/{item.citekey}")
            os.makedirs(loc_path, exist_ok=True)
            zenodo_get.download( ds_path, output_dir=loc_path ) #,   unzip=True) #, verbose=True)

            
            #loc_path = loc_path + "/EduRABSA_Dataset/2_annotated_dataset_files/pyabsa_dataset_format/ASQE/"
            dfs = load_dir(loc_path)
                #dfs = {k: transform(v) for k,v in dfs.items()}

        case "hesa":
            loc_path = os.path.expanduser(f"~/.cache/{item.repository.lower()}/{item.citekey}")
            os.makedirs(loc_path, exist_ok=True)
            files = os.listdir(loc_path)
            if files:
                dfs = load_dir(loc_path)
            else:
                print("no files in", loc_path, "download them manually and put there")
        
        case "manual":
            loc_path = os.path.expanduser(f"~/.cache/{item.repository.lower()}/{item.citekey}")
            os.makedirs(loc_path, exist_ok=True)
            files = os.listdir(loc_path)
            if files:
                dfs = load_dir(loc_path)
            else:
                print("no files in", loc_path, "download them manually and put there")
        

        case _ :
            print("skipping not implemented", item.citekey, "from", item.repository )

    if item.citekey in need_preprocess:
        transform = need_preprocess[item.citekey]
        dfs = {k: transform(v) for k,v in dfs.items()}

    return dfs


def load_dir(loc_path):
    dfs = {}
    files = os.listdir(loc_path)
    print("files:", files)
    for f in files:
        pth = os.path.join(loc_path, f)
        df = read_any(pth)
        if df is not None:
            dfs[f] = df
    return dfs


In [ ]:
itm = dss.loc[0]
itm

In [ ]:
# output_root = "tmp"
output_root ="~/Dropbox/Studies/M.Sc./Capstone/Datasets"
dfs = download(itm)

## Generate summaries

In [ ]:
def create_histogram(data_series):

    if np.issubdtype(data_series.dtype, np.number):
        pass
    elif np.issubdtype(data_series.dtype, np.datetime64):
        data_series = data_series.astype(np.int64) // 10**9  # Convert to seconds since epoch
    elif np.issubdtype(data_series.dtype, np.bool):
        data_series = data_series.astype(int)
    else:
        data_series = data_series.str.len()
        
    # Create a new plot
    fig, ax = plt.subplots(figsize=(1.0, 0.5), dpi=200)        # Plot the histogram
    nbins = min(10, data_series.nunique())
    ax.hist(data_series, bins=nbins, edgecolor='none', color='#4CAF50')
    
    # Hide axes and labels for a cleaner "sparkline" look
    #ax.set_xticks([])
    ax.set_yticks([])
    ax.set_frame_on(False)
    
    # Use an in-memory buffer
    buf = io.BytesIO()
    plt.savefig(buf, format='png', bbox_inches='tight', pad_inches=0.05)
    
    # Close the plot to free memory
    plt.close(fig)
    
    return  buf.getvalue()


In [ ]:
def create_base64_histogram(data_series):
    imgdata = create_histogram(data_series)
    # Encode the image to base64
    data_uri = base64.b64encode(imgdata).decode('utf-8')
    
    # Create an HTML image tag
    return f'<img src="data:image/png;base64,{data_uri}">'

In [ ]:
def create_png_histogram(data_series):
    imgdata = create_histogram(data_series)
    return imgdata


In [ ]:
df = dfs['train']

In [ ]:
hists_data = df.apply(lambda row: create_png_histogram(row), axis=0)

In [ ]:
hists_data

In [ ]:
def summary(df, hist_fmt = 'base64'):

    if hist_fmt == 'base64':
        hists = df.apply(lambda row: create_base64_histogram(row), axis=0)
    elif hist_fmt == 'png':
        hists = df.apply(lambda row: create_histogram(row), axis=0)

    #hists = 'Histogram' * df.shape[0]

    nonnans = df.shape[0] - df.isna().sum()
    nonnansPrc = (nonnans / df.shape[0] * 100).apply("{0:.2f}%".format)
    sam1 = df.sample(1, random_state=42).squeeze()
    sam2 = df.sample(1, random_state=495).squeeze()
    res = pd.DataFrame([sam1.index, df.dtypes.astype(str), nonnans,
                        nonnansPrc, df.nunique(), hists, sam1, sam2]).transpose()
    res.columns = ["Column", "data type", "non-null values", 
                   "non-null values %", "unique values","Hist", "example1", "example2"]
    res.sort_values([ "non-null values","unique values"],ascending=False, inplace=True)
    # let's abuse python's very lascivious OOP system
    res._repr_html_ = lambda: res.to_html(escape=False)
    return res

In [ ]:

def summary_hists(summary_df, ):

    hists_data = df.apply(lambda row: create_png_histogram(row), axis=0)

    link = f'![Histogram]({os.path.join(IMAGE_DIR, filename)})'

In [ ]:

def save_df_as_pretty_html(df, filename="output.html", index=True):
    pd.set_option("display.max_colwidth", None)
    # Convert newlines to <br> for HTML
    df_html_ready = df.copy()
    for col in df_html_ready.columns:
        df_html_ready[col] = df_html_ready[col].astype(str).str.replace('\n', '<br>', regex=False)

    # Generate styled HTML
    html = df_html_ready.to_html(
        escape=False,  # Needed to render <br>
        index=index,
        border=0,
        classes="styled-table"
    )

    # Add CSS styling
    style = """
    <style>
    .styled-table {
        border-collapse: collapse;
        margin: 25px 0;
        font-size: 16px;
        font-family: Arial, sans-serif;
        width: 100%;
        table-layout: auto; /* ✅ Let browser fit naturally */
    }
    .styled-table th, .styled-table td {
        border: 1px solid #dddddd;
        padding: 10px;
        vertical-align: top;
        text-align: left;
        overflow-wrap: break-word; /* ✅ Break inside words */
        white-space: pre-wrap; /* ✅ Honor \\n linebreaks */
    }
    .styled-table td {
        max-width: 600px; /* ✅ Avoid huge dream fields expanding table */
    }
    .styled-table th {
        background-color: #f2f2f2;
    }
    </style>
    """

    # Write full HTML document
    with open(filename, "w", encoding="utf-8") as f:
        f.write(f"<!DOCTYPE html><html><head>{style}</head><body>{html}</body></html>")

    print(f"✅ HTML table saved to: {filename}")


## Pipeline

In [ ]:
import re

def md_save(df, folder, tblname):
    os.makedirs(folder + "/img", exist_ok=True)
    tblname = os.path.splitext(tblname)[0]
    # save images
    for i,r in df.iterrows():

        imgname = f'img/{tblname}_{r.Column}.png'
        imgname = imgname.replace(" ","_").replace(":","_")

        with open(folder + "/" + imgname, 'wb') as f:
            f.write(r.Hist)
        r.Hist = f'![Histogram]({imgname})'

    #mcw = [10] * 5 + [None] + [40,40]
    
    md = df.to_markdown(index = False, tablefmt="pipe") # , maxcolwidths=mcw)
    md = re.sub(r'-{42,}', '-'*42, md)  # shorten columns width
    with open(folder + "/" + tblname + ".md" ,'w+') as f:
        f.write(md)

In [ ]:
summ = summs['test']
summ

In [ ]:
# This pipeline writes summaries of each table as separate md file under item's folder

def pipeline(item, force = False):
    #print(item.citekey)
    
    folder = output_root + "/" + item.citekey
    # if os.path.isdir(folder) :
    #     print("skipping existing item", folder, "from",  item.repository.lower() )
    #     return # None, folder
    
    dfs = download(item)
    summs = {}
    for dfname, df in dfs.items():
        dfname = os.path.splitext(dfname)[0]

        outpath = os.path.join(folder , dfname + '.md')

        if os.path.exists(outpath) and not force:
            print (f"Skipping summarization - already exists: '{outpath}'")
            continue

        try:
            summ = summary(df, hist_fmt='png')
        except Exception as e:
            summ = pd.DataFrame([["oops",b"oobs","Error during summarization", str(e)]], columns=["Column", "Hist", "Error", "Message"])
            warnings.warn(f"Error summarizing {dfname}: {e}")
        summs[dfname] = summ
        

        md_save(summ, folder, tblname=dfname)
        print("saved", outpath)

    return dfs, summs
        

In [ ]:
# This pipeline writes summaries of each table as headers under item's md file

def pipeline(item, force = False):
    #print(item.citekey)
    
    folder = output_root + "/" + item.citekey
    # if os.path.isdir(folder) :
    #     print("skipping existing item", folder, "from",  item.repository.lower() )
    #     return # None, folder
    
    dfs = download(item)
    summs = {}
    for dfname, df in dfs.items():
        dfname = os.path.splitext(dfname)[0]

        outpath = os.path.join(folder , dfname + '.md')

        if os.path.exists(outpath) and not force:
            print (f"Skipping summarization - already exists: '{outpath}'")
            continue

        try:
            summ = summary(df, hist_fmt='png')
        except Exception as e:
            summ = pd.DataFrame([["oops",b"oobs","Error during summarization", str(e)]], columns=["Column", "Hist", "Error", "Message"])
            warnings.warn(f"Error summarizing {dfname}: {e}")
        summs[dfname] = summ
        

        md_save(summ, folder, tblname=dfname)
        print("saved", outpath)

    return dfs, summs
        

In [ ]:
dss


In [ ]:
itm = dss.loc[0]

In [ ]:
dfs, summs =  pipeline(itm, True)

In [ ]:
summs.keys()

In [ ]:
summs['train']

In [ ]:
summ = summs['train']
summ.columns.to_list()

In [ ]:
[10] * 5 + [None] + [40,40]

In [ ]:
md_save(summ, "test.md")

## Run Pipeline for all items

In [ ]:
output_root

In [ ]:


for i, d in dss.iterrows():
    
    pipeline(d, True)

print("all done")

In [ ]:

wat = pd.read_csv(pth, )
wat

In [ ]:
summary(wat)